# Step 1 – Base district heating network (DHNx)

5 consumers, 2 heat producers, 2 pipe types (DN-32 and DN-50).

DHNx gets a set of *candidate* pipes (the streets of the tutorial network, which form a ring,
so there is more than one possible route) and decides

- which candidate pipes to build (the network topology), and
- which pipe type each built pipe gets,

so that the yearly cost (pipe annuities + heat production cost, including the heat lost in the
pipes) is as low as possible.

All assumed numbers (pipes, producers, temperatures, interest rate) are in `src/parameters.py`,
with their sources. The README explains the model in more detail.

Run with the **Python (dhnx_env)** kernel. The optimisation takes about 2–3 minutes.

In [ ]:
import os
import sys

# src/ lives in the project root, one level above this notebook
ROOT = os.path.abspath("..")
sys.path.insert(0, ROOT)
# make this environment's CBC solver findable, even if the kernel was started
# without activating the conda environment
os.environ["PATH"] = os.path.join(sys.prefix, "Library", "bin") + os.pathsep + os.environ["PATH"]

import pandas as pd
from IPython.display import Image, display

from src import network_tools as nt
from src import parameters as par
from src import plotting

## 1. Settings

Which pipe types, producers and demand profiles Step 1 uses, and which of the tutorial's
8 house spots the 5 consumers are connected to.

In [ ]:
STEP = "step1"
NET_DIR = os.path.join(nt.ROOT, "inputs", STEP, "network")
INV_DIR = os.path.join(nt.ROOT, "inputs", STEP, "investment_options")
OUT_DIR = os.path.join(nt.ROOT, "results", STEP)

PIPES = ["DN-32", "DN-50"]
PRODUCERS = par.PRODUCERS_STEP1

# Demand profile (column in the Excel file) of each consumer.
# Loads 2-6 are the five distinct small profiles in the dataset (most other
# columns are copies of these with a constant added).
CONSUMER_PROFILES = {"0": "Load 2", "1": "Load 3", "2": "Load 4",
                     "3": "Load 5", "4": "Load 6"}

# Our consumer id -> tutorial consumer id (house spot), spread around the ring
CONSUMER_SPOTS = {"0": "0",   # at fork 2
                  "1": "2",   # at fork 3
                  "2": "4",   # at fork 7
                  "3": "5",   # at fork 8
                  "4": "6"}   # at fork 10

## 2. Heat demand

Hourly demand [kW] of the five consumers, straight from the course data
(`data/Demand_profiles_heating.xlsx`).

In [ ]:
demand_all = nt.load_demand()
demand = pd.DataFrame({cid: demand_all[col].values
                       for cid, col in CONSUMER_PROFILES.items()})

print(f"Coincident peak (all five together): {demand.sum(axis=1).max():.0f} kW")
pd.DataFrame({"profile": CONSUMER_PROFILES,
              "peak [kW]": demand.max().round(1),
              "yearly demand [MWh]": (demand.sum() / 1000).round(1)})

## 3. Network layout

The course DHNx tutorial's *introduction* network (`DNHx Tutorial/introduction/twn_data`):
2 producers, 11 forks in a ring and the house connections. These are the candidate pipes
DHNx can choose from, with the lengths given in the tutorial.

In [ ]:
LAYOUT_DIR = os.path.join(nt.ROOT, "DNHx Tutorial", "introduction", "twn_data")
producers, forks, consumers, streets, connections, lengths =     nt.load_tutorial_layout(LAYOUT_DIR, CONSUMER_SPOTS)

pd.DataFrame([{"from": a, "to": b, "length [m]": l} for (a, b), l in lengths.items()])

## 4. Pipe types and producers

Investment cost and heat loss come from the tutorial (`discrete_DN_numbers/Case 1`).
`fix_costs` is the investment cost turned into a yearly cost (4 %, 40 years).
Capacity is calculated from ΔT = 40 K and a maximum velocity (see `src/parameters.py`).

In [ ]:
pipe_table = pd.DataFrame(par.pipe_table(PIPES))
pipe_table[["label_3", "cap_max", "v_max_m_s", "invest_eur_per_m", "fix_costs", "l_factor_fix"]]

In [ ]:
pd.DataFrame(PRODUCERS).T

## 5. Optimisation

Two stages, so it solves in minutes rather than hours:

1. **Design:** DHNx chooses which pipes to build and their types, using 12 *design hours*
   (each consumer's peak hour plus the hours with the highest total demand).
2. **Operation:** the chosen pipes are fixed and the network is run for all 8760 hours, to get
   the yearly heat losses, producer output and costs.

The DHNx input files are written to `inputs/step1/` (open them in Excel to see exactly what the
model gets).

In [ ]:
design, operation, pipe_table, hours = nt.design_then_operate(
    NET_DIR, INV_DIR, producers, forks, consumers, streets, connections,
    demand, PIPES, PRODUCERS, lengths)

## 6. Results

Tables are also saved in `results/step1/`.

In [ ]:
key, built, prod_summary, by_type = nt.summarise(
    design, operation, pipe_table, PRODUCERS, demand, OUT_DIR)
built

In [ ]:
by_type

In [ ]:
prod_summary

In [ ]:
pd.Series(key, name="value").to_frame()

## 7. Plots

In [ ]:
names = {"producers": {i: p["name"] for i, p in PRODUCERS.items()},
         "consumers": CONSUMER_PROFILES}
candidates = pd.read_csv(os.path.join(NET_DIR, "pipes.csv"))
plotting.plot_network(producers, forks, consumers, candidates, built, names,
                      "Step 1 - optimised network",
                      os.path.join(OUT_DIR, "network.png"))
display(Image(os.path.join(OUT_DIR, "network.png")))

In [ ]:
prod = pd.read_csv(os.path.join(OUT_DIR, "producer_output_hourly.csv"), index_col=0)
prod.columns = prod.columns.astype(str)
plotting.plot_producers(prod, names["producers"], demand.sum(axis=1),
                        "Step 1 - heat supplied by each producer",
                        os.path.join(OUT_DIR, "producers.png"))
display(Image(os.path.join(OUT_DIR, "producers.png")))